In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#installing dependencies

import numpy as np
import argparse
import cv2 as cv
import os
import matplotlib.pyplot as plt

In [ ]:
#opening the pre-trained model files correctly

model_path = '/kaggle/input/colorization-caffe-model/colorization_release_v2.caffemodel'
points_path = '/kaggle/input/colorization-caffe-model/pts_in_hull.npy'
prototxt_path = '/kaggle/input/colorization-caffe-model/colorization_deploy_v2.prototxt'

# Load the cluster centers
pts_in_hull = np.load(points_path)

# Load the network
net = cv.dnn.readNetFromCaffe(prototxt_path, model_path)

In [ ]:
#OpenCV Colorization Model 

# Constants
W_in = 224
H_in = 224
imshowSize = (640, 480)

#different inputs for the model to make life easier
input_1 = '/kaggle/input/test-image-mia'
input_2 = '/kaggle/input/train-data-mia'
input_3 = '/kaggle/input/grayscale-dataset'

# Define paths manually (no argparse in Kaggle)
class Args:
    input_dir = input_2
    output_dir = '/kaggle/working/colorized_images'  # directory to save colorized images
    prototxt = '/kaggle/input/colorization-caffe-model/colorization_deploy_v2.prototxt'
    caffemodel = '/kaggle/input/colorization-caffe-model/colorization_release_v2.caffemodel'
    kernel = '/kaggle/input/colorization-caffe-model/pts_in_hull.npy'

args = Args()

# Load model
net = cv.dnn.readNetFromCaffe(args.prototxt, args.caffemodel)

# Load the kernel (pts_in_hull)
pts_in_hull = np.load(args.kernel)

# Populate cluster centers as 1x1 convolution kernel
pts_in_hull = pts_in_hull.transpose().reshape(2, 313, 1, 1)
net.getLayer(net.getLayerId('class8_ab')).blobs = [pts_in_hull.astype(np.float32)]
net.getLayer(net.getLayerId('conv8_313_rh')).blobs = [np.full([1, 313], 2.606, np.float32)]

# Function to process and predict colorization for each image
def process_image(img_path, net):
    # Load image
    frame = cv.imread(img_path)
    img_rgb = (frame[:, :, [2, 1, 0]] * 1.0 / 255).astype(np.float32)
    img_lab = cv.cvtColor(img_rgb, cv.COLOR_RGB2Lab)
    img_l = img_lab[:, :, 0]  # pull out L channel
    (H_orig, W_orig) = img_rgb.shape[:2] 
    # Resize and preprocess
    img_rs = cv.resize(img_rgb, (W_in, H_in), interpolation=cv.INTER_CUBIC)  # or INTER_LANCZOS4
    img_lab_rs = cv.cvtColor(img_rs, cv.COLOR_RGB2Lab)
    img_l_rs = img_lab_rs[:, :, 0]
    img_l_rs -= 45

    # Predict color
    net.setInput(cv.dnn.blobFromImage(img_l_rs))
    ab_dec = net.forward()[0, :, :, :].transpose((1, 2, 0))
    ab_dec_us = cv.resize(ab_dec, (W_orig, H_orig))
    img_lab_out = np.concatenate((img_l[:, :, np.newaxis], ab_dec_us), axis=2)
    img_bgr_out = np.clip(cv.cvtColor(img_lab_out, cv.COLOR_Lab2BGR), 0, 1)

    # Convert to uint8 and save the output image
    img_bgr_out = (img_bgr_out * 255).astype(np.uint8)
    return img_bgr_out, os.path.basename(img_path)

# Function to save and display multiple output images
def save_and_display_images(images, output_dir, titles=None):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Save each image to the output directory and display them
    for idx, img in enumerate(images):
        img_filename = os.path.join(output_dir, f"colorized_{titles[idx]}")
        cv.imwrite(img_filename, img)
        
        # Convert BGR to RGB for display
        img_rgb_out = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        
        # Display image using matplotlib
        plt.imshow(img_rgb_out)
        plt.title(f'Colorized - {titles[idx]}')
        plt.axis('off')
        plt.show()

# Loop through all images in input directory
input_images = [os.path.join(args.input_dir, f) for f in os.listdir(args.input_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

output_images = []  # Store colorized output images
titles = []  # Store titles for the images

for img_path in input_images:
    # Process the image
    img_out, title = process_image(img_path, net)

    # Append the output image and title to the lists
    output_images.append(img_out)
    titles.append(title)

# Save and display all the colorized images
save_and_display_images(output_images, args.output_dir, titles)